# Recovery of the missing HR-VPP tile **T34UGV** (Copernicus Vegetation Phenology and Productivity)

**Purpose.** MONIFUN WP3 needs the Copernicus HR-VPP parameters `AMPL`, `LENGTH`, `RSLOPE`, `TPROD`
(season 1, 2017–2023) at NFI plots. The download workflow in the MONIFUN wiki
(`download_copernicus_pheno.py`, WEkEO HDA) returns nothing for one Sentinel-2 tile in western Ukraine:
**T34UGV** (≈ 23.7–25.3° E, 48.6–49.6° N; Ivano-Frankivsk, Kalush, Stryi area). The tile is absent from the
operational catalogue for *every* year 2017–2025, while all neighbours (T34UFV, T34UGA, T34UGU, T35ULP) exist.
It therefore cannot be "re-downloaded"; it has to be **reconstructed** from Sentinel-2.

**What this notebook does**

| Step | What | Needs |
|---|---|---|
| 1 | Confirms the gap in the CLMS/VITO catalogue | nothing (anonymous) |
| 2 | MONIFUN-compatible WEkEO download; neighbours for validation | WEkEO account (Colab Secrets) |
| 3 | Reconstructs season-1 VPP parameters in Google Earth Engine (HR-VPP PPI formula, 25 %/15 % thresholds) | Earth Engine project |
| 4 | Cross-checks the Earth Engine code against an independent NumPy implementation | Earth Engine |
| 5 | **Validates** against official HR-VPP pixels in the ~10 km overlap strips with T34UFV/T34UGA/T34UGU | Steps 2 + 3 |
| 6 | Exports the full 10 m tile for all years and converts it to the MONIFUN folder layout with official encoding | Earth Engine, Drive |
| 7 | Reproducibility checks and a written report | – |

> **Important.** Reconstructed files are *not* official CLMS products. They are tagged as such
> (`SOURCE` GeoTIFF tag, `_RECON_` in the file name) and must be reported as gap-filled in any publication.

Method and validation protocol: see `docs/METHOD.md` and `docs/VALIDATION.md` in the project folder / GitHub repository.

---
*Українською коротко:* ноутбук підтверджує, що тайла T34UGV немає в каталозі HR-VPP, відтворює для нього
параметри AMPL, LENGTH, RSLOPE, TPROD з Sentinel-2 за методикою HR-VPP і перевіряє точність на ~10-км смугах
перекриття з офіційними сусідніми тайлами. Результат кладеться у структуру папок MONIFUN.

In [ ]:
#@title 0. Parameters { display-mode: "form" }
EE_PROJECT = ""  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/HRVPP_T34UGV_recovery"  #@param {type:"string"}
YEARS = "2017-2023"  #@param {type:"string"}
PRODUCTS = "AMPL,LENGTH,RSLOPE,TPROD"  #@param {type:"string"}
VALIDATION_YEARS = "2017-2023"  #@param {type:"string"}
N_VALIDATION_POINTS = 3000  #@param {type:"integer"}
DOWNLOAD_NEIGHBOURS = True  #@param {type:"boolean"}
START_FULL_EXPORTS = False  #@param {type:"boolean"}

def parse_years(s):
    out = []
    for part in s.replace(" ", "").split(","):
        if "-" in part:
            a, b = map(int, part.split("-")); out += list(range(a, b + 1))
        elif part:
            out.append(int(part))
    return out

years = parse_years(YEARS)
val_years = parse_years(VALIDATION_YEARS)
products = [p.strip().upper() for p in PRODUCTS.split(",") if p.strip()]
print("years:", years, "| validation:", val_years, "| products:", products)

In [ ]:
#@title 0.1 Environment: Drive, code, libraries
import os, sys, subprocess, json, time, datetime as dt
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT); ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hda>=2", "rasterio", "earthengine-api"], check=True)
sys.path.insert(0, str(ROOT / "src"))
import importlib, hrvpp_recon
from hrvpp_recon import config, catalogue, wekeo, phenology_np
print("hrvpp_recon", hrvpp_recon.__version__, "from", hrvpp_recon.__file__)
OFFICIAL = ROOT / "Copernicus_Phenology_official"     # downloaded CLMS files (MONIFUN layout)
RECON = ROOT / "Copernicus_Phenology"                  # reconstructed T34UGV (MONIFUN layout)
REPORT = ROOT / "reports"; REPORT.mkdir(exist_ok=True)
RUN_LOG = {"started": dt.datetime.utcnow().isoformat() + "Z", "years": years, "products": products}

## 1. Is the tile really missing? (anonymous catalogue check)

The VITO OpenSearch catalogue that backs WEkEO and the CLMS viewer is queried for every year. For each year we list
which Sentinel-2 tiles intersect the gap footprint. `available = False` for T34UGV in every year confirms that the
gap is a production gap, not a download problem.

In [ ]:
#@title 1. Catalogue availability of T34UGV (VPP and PPI Seasonal Trajectories)
import pandas as pd
avail = catalogue.tile_availability("T34UGV", config.GAP_BBOX_LONLAT, range(2017, 2026))
display(avail)
centre = [catalogue.point_has_product(24.6, 49.1, y) for y in range(2017, 2026)]
centre_st = [catalogue.point_has_product(24.6, 49.1, y, collection=config.VITO_COLLECTION_ST_UTM) for y in range(2017, 2026)]
chk = pd.DataFrame({"year": range(2017, 2026), "VPP products at 24.6E 49.1N": centre, "PPI-ST products at 24.6E 49.1N": centre_st})
display(chk)
avail.to_csv(REPORT / "01_catalogue_availability.csv", index=False)
RUN_LOG["gap_confirmed"] = bool((~avail["available"]).all())
print("T34UGV absent in all years:", RUN_LOG["gap_confirmed"])

## 2. WEkEO HDA (MONIFUN workflow)

Credentials are read from **Colab Secrets** (🔑 icon in the left bar): add `WEKEO_USER` and `WEKEO_PASSWORD`
and enable notebook access. Nothing is stored in the notebook.

* 2a lists what HDA returns for the gap footprint (the MONIFUN query, unchanged).
* 2b downloads only the three official neighbours that overlap T34UGV, for the validation years
  (≈ 0.25 GB per file). Files land in `Copernicus_Phenology_official/<PRODUCT>/<YEAR>/` exactly as in MONIFUN.

In [ ]:
#@title 2a. What does the MONIFUN HDA query return for the gap?
HAS_WEKEO = wekeo.load_credentials()
print("WEkEO credentials found:", HAS_WEKEO)
if HAS_WEKEO:
    rows = []
    for y in val_years:
        ids = wekeo.search_ids(y, "AMPL", config.GAP_BBOX_LONLAT)
        rows.append({"year": y, "n_results": len(ids), "T34UGV_present": any("T34UGV" in i for i in ids),
                     "tiles": ", ".join(sorted({i.split("_")[3] for i in ids if i.startswith("VPP_")}))})
    hda_df = pd.DataFrame(rows); display(hda_df)
    hda_df.to_csv(REPORT / "02_hda_search.csv", index=False)
else:
    print("Skipped: add WEKEO_USER / WEKEO_PASSWORD in Colab Secrets to enable steps 2 and 5.")

In [ ]:
#@title 2b. Download official neighbours (T34UFV, T34UGA, T34UGU) for validation
if HAS_WEKEO and DOWNLOAD_NEIGHBOURS:
    summary = []
    for nb, (lon, lat) in config.NEIGHBOUR_PROBES.items():
        for y in val_years:
            for prod in products:
                if list((OFFICIAL / prod / str(y)).glob(f"*{nb}*_s1_{prod}.tif")):
                    continue
                r = wekeo.download_one(year=y, product=prod, outdir=OFFICIAL, bbox=wekeo.probe_bbox(lon, lat))
                summary.append({**r.__dict__, "neighbour": nb})
    if summary:
        pd.DataFrame(summary).to_csv(REPORT / "02_download_summary.csv", index=False)
    n = wekeo.write_checksums(OFFICIAL, REPORT / "02_official_checksums.csv")
    print(n, "official files present; checksums written.")

## 3. Reconstruction in Google Earth Engine

Sentinel-2 L2A (L1C only where no L2A scene exists) → Cloud Score+ masking → **Plant Phenology Index** with the
HR-VPP ATBD formula → 10-day composites → winter base-level filling and linear interpolation → binomial smoothing →
season metrics with the HR-VPP thresholds (SOS at 25 %, EOS at 15 % of amplitude; season assigned to the year of its peak).

If `EE_PROJECT` is empty, Earth Engine uses the project linked to your account.

In [ ]:
#@title 3. Initialise Earth Engine and build the reconstruction
import ee
try:
    ee.Initialize(project=EE_PROJECT or None)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT or None)
from hrvpp_recon import gee_recon, fastval
importlib.reload(gee_recon); importlib.reload(fastval)
P = config.ReconParams()
recon = {y: gee_recon.reconstruct_year(y, p=P) for y in sorted(set(years) | set(val_years))}
y0 = val_years[0]
print("Sentinel-2 scenes in the", y0, "window:", recon[y0]["n_scenes"].getInfo())
print("Parameters:", P)

In [ ]:
#@title 3.1 Quick look (200 m preview of the reconstructed tile and its official neighbours' extent)
from IPython.display import Image, display
region = gee_recon.tile_geometry()
vis = {"AMPL": (0, 2.0), "LENGTH": (60, 260), "TPROD": (0, 500), "RSLOPE": (0, 0.05)}
for prod in products:
    lo, hi = vis.get(prod, (0, 1))
    url = recon[y0]["metrics"].select(prod).getThumbURL({
        "region": region, "dimensions": 512, "min": lo, "max": hi, "crs": "EPSG:32634",
        "palette": ["440154", "3b528b", "21918c", "5ec962", "fde725"]})
    print(prod, y0, f"[{lo} .. {hi}]"); display(Image(url=url))
cov = recon[y0]["metrics"].select("AMPL").mask().reduceRegion(ee.Reducer.mean(), region, 500, maxPixels=1e9).getInfo()
l1c = recon[y0]["metrics"].select("L1C_SHARE").reduceRegion(ee.Reducer.mean(), region, 1000, maxPixels=1e9).getInfo()
print("share of tile with a valid season:", round(100 * cov["AMPL"], 1), "% | share of L1C observations:", l1c)

## 4. Implementation cross-check (Earth Engine vs NumPy reference)

The smoothed PPI series is sampled at 300 random pixels and the season metrics are recomputed with the independent
NumPy implementation (`phenology_np.py`, unit-tested on synthetic double-logistic curves with known answers).
Differences should be at float32 rounding level (≈1e-5); any larger difference means the two implementations diverged.

In [ ]:
#@title 4. Earth Engine vs NumPy
xc = fastval.crosscheck_numpy(recon[y0], y0)
display(xc)
xc.to_csv(REPORT / "04_crosscheck_ee_numpy.csv", index=False)
RUN_LOG["crosscheck_max_abs_diff"] = xc.set_index("metric")["max_abs_diff"].to_dict()

## 5. Validation against official HR-VPP pixels (overlap strips)

T34UGV overlaps T34UFV (west), T34UGA (north) and T34UGU (south) by ~10 km. There, official and reconstructed
pixels share the same 10 m grid. `N_VALIDATION_POINTS` random pixel centres per strip are compared.

Reported: n, Pearson r, bias (reconstructed − official), MAE, RMSE, relative RMSE, OLS slope/intercept, and the share
of pixels with a valid season in each source.

In [ ]:
#@title 5. Point-based validation
import matplotlib.pyplot as plt, numpy as np
val_tables = []
for y in val_years:
    if not list(OFFICIAL.rglob(f"VPP_{y}_*.tif")):
        print(y, ": no official files – run step 2b first"); continue
    v = fastval.fast_validation(recon[y]["metrics"], y, OFFICIAL, products, n=N_VALIDATION_POINTS)
    val_tables.append(v)
if val_tables:
    val = pd.concat(val_tables, ignore_index=True)
    pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
    display(val)
    val.to_csv(REPORT / "05_validation_points.csv", index=False)
    pooled = val.groupby("product")[["r", "bias", "mae", "rmse", "rrmse_pct"]].median()
    display(pooled)
    RUN_LOG["validation_median_by_product"] = pooled.round(4).to_dict(orient="index")

In [ ]:
#@title 5.1 Scatter plots (pooled over strips, first validation year)
if val_tables:
    y = val_years[0]
    fig, axes = plt.subplots(1, len(products), figsize=(4.2 * len(products), 4))
    for ax, prod in zip(np.atleast_1d(axes), products):
        refs, ests = [], []
        for nb_id, nb in config.NEIGHBOURS.items():
            f = fastval.find_official(OFFICIAL, prod, y, nb_id)
            if f is None: continue
            xy = fastval.strip_points(nb, n=N_VALIDATION_POINTS)
            refs.append(fastval.sample_official(f, xy))
            ests.append(fastval.sample_ee(recon[y]["metrics"], xy, 32634, [prod])[prod].to_numpy(float))
        if not refs: continue
        a, b = np.concatenate(refs), np.concatenate(ests); m = np.isfinite(a) & np.isfinite(b)
        ax.hexbin(a[m], b[m], gridsize=60, mincnt=1, bins="log", cmap="viridis")
        lim = [np.nanpercentile(a[m], 1), np.nanpercentile(a[m], 99)]
        ax.plot(lim, lim, "r--", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_title(f"{prod} {y}  r={np.corrcoef(a[m], b[m])[0,1]:.2f}  n={m.sum()}")
        ax.set_xlabel("official HR-VPP"); ax.set_ylabel("reconstructed")
    plt.tight_layout(); plt.savefig(REPORT / f"05_scatter_{y}.png", dpi=130); plt.show()

## 6. Full-tile export and MONIFUN layout

Starts one Earth Engine export per year (10 m, exact Sentinel-2 grid of T34UGV, EPSG:32634, Int16). Exports go to the
Drive folder with the same name as `DRIVE_ROOT`'s last component. Each year takes roughly 1–3 h on Earth Engine.
Re-run 6.1 when the tasks have finished: it converts the exports to official HR-VPP encoding and writes
`Copernicus_Phenology/<PRODUCT>/<YEAR>/VPP_<YEAR>_S2_T34UGV-010m_RECON_s1_<PRODUCT>.tif`.

In [ ]:
#@title 6. Start exports (only if START_FULL_EXPORTS)
tasks = {}
if START_FULL_EXPORTS:
    for y in years:
        tasks[y] = gee_recon.export_year(y, folder=ROOT.name, p=P)
        print(y, tasks[y].id)
    json.dump({y: t.id for y, t in tasks.items()}, open(REPORT / "06_export_tasks.json", "w"))
else:
    print("Exports not started (tick START_FULL_EXPORTS).")
for t in ee.data.getTaskList()[:15]:
    if t.get("description", "").startswith("RECON_T34UGV"):
        print(t["description"], t["state"], t.get("error_message", ""))

In [ ]:
#@title 6.1 Fetch exports, calibrate on the overlap strips, write the MONIFUN layout
# Earth Engine writes exports to the Drive of the *Earth Engine* account. If that differs from the
# Colab Drive, fetch them with the Earth Engine credentials (needs the Drive API in EE_PROJECT).
from hrvpp_recon import postprocess, validate, eedrive
LOCAL = Path("/content/recon")
done = sorted(ROOT.glob("RECON_VPP_*_S2_T34UGV-010m_s1*.tif"))
if not done and EE_PROJECT:
    done = eedrive.fetch("RECON_VPP_", LOCAL, EE_PROJECT)
print(len(done), "export file(s):", [p.name for p in done])
written, cal_log = [], {}
for f in done:
    y = int(f.name.split("_")[2])
    fits, loso, raw = validate.strip_calibration(f, OFFICIAL, y, products, list(config.NEIGHBOURS))
    cal_log[y] = {"fits": fits, "loso": loso}
    for prod in products:
        off = next(iter(sorted(OFFICIAL.glob(f"{prod}/*/*_s1_{prod}.tif"))), None)
        cal = fits.get(prod) if prod in config.CALIBRATED_PRODUCTS else None
        if prod in config.CALIBRATED_PRODUCTS and cal is None:
            print(f"WARNING {y} {prod}: no official neighbours downloaded for this year -> written uncalibrated")
        written.append(postprocess.gee_to_product(f, prod, y, RECON, "T34UGV", official_tif=off, calibration=cal))
json.dump(cal_log, open(REPORT / "06_calibration_log.json", "w"), indent=1, default=float)
print("\n".join(str(w) for w in written))


In [ ]:
#@title 6.2 Full-overlap validation on exported files (all pixels of the strips)
from hrvpp_recon import validate
rows = []
for f in sorted(ROOT.glob("RECON_VPP_*_S2_T34UGV-010m_s1*.tif")):
    y = int(f.name.split("_")[2])
    for nb in config.NEIGHBOURS:
        for prod in products:
            off = fastval.find_official(OFFICIAL, prod, y, nb)
            if off is None: continue
            a, b = validate.overlap_arrays(f, prod, off)
            rows.append({"year": y, "neighbour": nb, "product": prod, **validate.metrics(a, b),
                         **validate.coverage_stats(off, f, prod)})
if rows:
    full = pd.DataFrame(rows); display(full)
    full.to_csv(REPORT / "06_validation_full_overlap.csv", index=False)

## 7. Reproducibility and report

* Earth Engine sampling is repeated with the same seed: values must be identical.
* Checksums of the official downloads are stored (`02_official_checksums.csv`); a second download can be compared
  file by file.
* A machine-readable run log and a short Markdown report are written to `reports/`.

In [ ]:
#@title 7. Reproducibility + report
xy = fastval.strip_points(config.NEIGHBOURS["T34UFV"], n=200, seed=99)
s1 = fastval.sample_ee(recon[y0]["metrics"], xy, 32634, products)
s2 = fastval.sample_ee(recon[y0]["metrics"], xy, 32634, products)
same = all(np.array_equal(s1[p].to_numpy(float), s2[p].to_numpy(float), equal_nan=True) for p in products)
print("Repeated Earth Engine sampling identical:", same)
RUN_LOG.update({"ee_repeat_identical": bool(same), "finished": dt.datetime.utcnow().isoformat() + "Z",
                "params": P.__dict__})
json.dump(RUN_LOG, open(REPORT / "run_log.json", "w"), indent=2, default=str)
lines = ["# T34UGV reconstruction – run report", "", f"Run: {RUN_LOG['started']} – {RUN_LOG['finished']}", "",
         f"* Gap confirmed in catalogue for all years: **{RUN_LOG.get('gap_confirmed')}**",
         f"* EE vs NumPy max abs differences: {RUN_LOG.get('crosscheck_max_abs_diff')}",
         f"* Repeated EE sampling identical: **{same}**", ""]
if "validation_median_by_product" in RUN_LOG:
    lines += ["## Validation (median over strips and years)", "", "| product | r | bias | MAE | RMSE | rRMSE % |", "|---|---|---|---|---|---|"]
    for k, v in RUN_LOG["validation_median_by_product"].items():
        lines.append(f"| {k} | {v['r']:.3f} | {v['bias']:.4f} | {v['mae']:.4f} | {v['rmse']:.4f} | {v['rrmse_pct']:.1f} |")
(REPORT / "REPORT.md").write_text("\n".join(lines))
print("\n".join(lines))